# Stage 2: Under-Provisioning Deep Analysis
**Cloud Resource Optimisation Thesis — Frankfurt School of Finance and Management**  
**Author:** Prarthana Govindaraj  
**Date:** May 2026  

---

## Research Questions
1. Who are the under-provisioned jobs — what does their profile look like?
2. What signals most reliably indicate a job will be under-provisioned?
3. At what thresholds does under-provisioning risk spike?
4. Can we predict the severity of under-provisioning, not just its presence?
5. What corrected allocation should we recommend per job?

## Structure
- Section 0: Load Stage 1b outputs
- Section 1: Deep profile of under-provisioned jobs
- Section 2: Signal analysis and threshold detection
- Section 3: Severity prediction model
- Section 4: Corrected allocation engine
- Section 5: Summary and findings

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import json
from pathlib import Path

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries imported successfully")

Libraries imported successfully


In [2]:
PROCESSED_DIR = Path('/Users/prarthanagovindaraj/Desktop/Cloud_Resource_Optimisation_thesis/data/processed')

# Load full dataset with all features and labels from Stage 1b
df = pd.read_csv(PROCESSED_DIR / 'stage1b_merged_clean.csv')

# Load Stage 1b summary
with open(PROCESSED_DIR / 'stage1b_summary.json', 'r') as f:
    summary = json.load(f)

print("=" * 60)
print("STAGE 1B OUTPUTS LOADED")
print("=" * 60)
print(f"\nDataset shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"\nColumns: {list(df.columns)}")
print(f"\nStage 1b summary:")
print(f"  Total jobs            : {summary['total_jobs']:,}")
print(f"  Under-provisioned     : {summary['under_provisioned_count']:,} ({summary['under_provisioned_pct']:.1f}%)")
print(f"  RF Recall (Under)     : {summary['rf_recall_under']:.3f}")
print(f"  RF Precision (Under)  : {summary['rf_precision_under']:.3f}")
print(f"  RF F1 (Under)         : {summary['rf_f1_under']:.3f}")
print(f"  RF ROC-AUC            : {summary['rf_roc_auc']:.3f}")
print(f"\nClass distribution from Stage 1b:")
for cls, count in summary['class_distribution'].items():
    pct = count / summary['total_jobs'] * 100
    print(f"  {cls:<20}: {count:>6,} ({pct:.1f}%)")

STAGE 1B OUTPUTS LOADED

Dataset shape: 11,644 rows x 19 columns

Columns: ['job_name', 'plan_cpu_cores', 'task_count', 'cpu_avg_peak', 'cpu_max_peak', 'instance_count', 'cpu_avg_mean', 'cpu_avg_std', 'cpu_avg_max', 'util_ratio_avg', 'util_ratio_peak', 'provisioning_class', 'is_underprovision', 'plan_cpu_per_task', 'instance_to_task_ratio', 'is_multi_task', 'is_large_job', 'size_bucket', 'cpu_size_bucket_encoded']

Stage 1b summary:
  Total jobs            : 11,644
  Under-provisioned     : 5,149 (44.2%)
  RF Recall (Under)     : 0.787
  RF Precision (Under)  : 0.818
  RF F1 (Under)         : 0.802
  RF ROC-AUC            : 0.908

Class distribution from Stage 1b:
  severely_under      :  3,225 (27.7%)
  under               :  1,924 (16.5%)
  efficient           :  3,206 (27.5%)
  over                :  3,289 (28.2%)


In [3]:
print("=" * 60)
print("DATA QUALITY CHECK")
print("=" * 60)

print(f"\nNull values:")
nulls = df.isnull().sum()
if nulls.sum() == 0:
    print("  None found")
else:
    print(nulls[nulls > 0])

print(f"\nKey column dtypes:")
key_cols = ['job_name', 'plan_cpu_cores', 'cpu_max_peak', 'cpu_avg_peak',
            'util_ratio_peak', 'util_ratio_avg', 'task_count',
            'instance_count', 'provisioning_class', 'is_underprovision']
for col in key_cols:
    if col in df.columns:
        print(f"  {col:<25}: {df[col].dtype}")

print(f"\nProvisioning class value counts:")
print(df['provisioning_class'].value_counts())

print(f"\nutil_ratio_peak stats:")
print(df['util_ratio_peak'].describe().round(3))

# Confirm split
under = df[df['is_underprovision'] == 1]
not_under = df[df['is_underprovision'] == 0]
print(f"\nWorking splits:")
print(f"  Under-provisioned     : {len(under):,}")
print(f"  Not under-provisioned : {len(not_under):,}")
print(f"\nReady for Section 1 — Deep Profile")

DATA QUALITY CHECK

Null values:
cpu_avg_std    1525
dtype: int64

Key column dtypes:
  job_name                 : int64
  plan_cpu_cores           : float64
  cpu_max_peak             : float64
  cpu_avg_peak             : float64
  util_ratio_peak          : float64
  util_ratio_avg           : float64
  task_count               : int64
  instance_count           : int64
  provisioning_class       : str
  is_underprovision        : int64

Provisioning class value counts:
provisioning_class
over              3289
severely_under    3225
efficient         3206
under             1924
Name: count, dtype: int64

util_ratio_peak stats:
count    11644.000
mean         1.625
std          2.343
min          0.019
25%          0.412
50%          0.868
75%          1.790
max         31.000
Name: util_ratio_peak, dtype: float64

Working splits:
  Under-provisioned     : 5,149
  Not under-provisioned : 6,495

Ready for Section 1 — Deep Profile
